## Extract Information for Joint Ontology Workshop (JOWO) and (FOIS)

In [16]:
import sys
import os

# Gehe 3 Ebenen nach oben ins Root-Verzeichnis (LiteraturResearcher/wo 'src' liegt)
sys.path.append(os.path.abspath('../../../..'))

import pandas as pd 
import numpy as np 
import time
from datetime import datetime
import re
from typing import List, Dict, Optional

# Die neuen Importe aus der 'src' Bibliothek:
from litresearch.extractors.DBLP_Extractor import DBLPConferenceExtractor
from litresearch import IOSPressDownloader
from litresearch.utils.clean import clean_html_entities
from litresearch.extractors.pdf_extractor import PDFExtractor



In [9]:
conference_dict = {
    "JOWO": [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    "FOIS": [2016, 2018, 2020, 2021, 2023, 2024, 2025]
}

## 1. Extract Paper Information from DBLP

In [ ]:
def get_papers_year_with_retry(
    year: int,
    extractor,
    max_retries: int = 4,
    backoff_base: float = 10.0,
    conference = ""
) -> pd.DataFrame:
    for attempt in range(1, max_retries + 1):
        try:
            papers = extractor.fetch_conference_papers(
                conference_query=f"stream:conf/{conference.lower()}:",
                venue_filter=[conference],
                venue_filter_mode="contains",
                year_start=year,
                year_end=year,
                verbose=True,
                console_output=False
            )
            if papers:
                df = pd.DataFrame(papers)
                return df[df['type'] != 'Editorship'].copy()
            return None

        except Exception as e:
            wait = backoff_base * attempt  # 10s, 20s, 30s, 40s
            extractor._log(
                f"  ⚠️ Attempt {attempt}/{max_retries} failed for {year}: {e}",
                console=True
            )
            if attempt < max_retries:
                extractor._log(f"  ⏳ Retrying in {wait}s...", console=True)
                time.sleep(wait)
            else:
                extractor._log(f"  ✗ Giving up on {year} after {max_retries} attempts.", console=True)
                return None


def get_res(year_list: List[int], conference: str, log_file: str) -> Dict[int, pd.DataFrame]:
    extractor = DBLPConferenceExtractor(log_file=log_file)
    results = {}

    for year in year_list:
        extractor._log(f"\n>>> Processing {conference} {year}", console=True)
        df = get_papers_year_with_retry(year, extractor, conference=conference)
        if df is not None and len(df) > 0:
            results[year] = df
        time.sleep(5)  # erhöht von 2s auf 5s zwischen Jahren

    return results

In [ ]:
ls_results = []

for conference, year_list in conference_dict.items():
    print(f"Verarbeite Konferenz: {conference}")
    log_file = f"dblp_extraction_{conference}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

    res = get_res(year_list, conference, log_file)
    ls_results.append(res)

    print(f"\n✓ {conference}: {len(res)}/{len(year_list)} Jahre gefunden")
    print(f"  Total Papers: {sum(len(df) for df in res.values())}")
    print(f"  Log gespeichert: {log_file}")

In [ ]:
df_all = pd.concat(
    [df for res in ls_results for df in res.values()],
    ignore_index=True
)
df_html_clean = clean_html_entities(df_all)

In [12]:
path = os.path.join("..","..","..", "data", "raw", "conferences","ontology","jowo_fois.csv")
df_html_clean = df_html_clean[(df_html_clean['venue'] == "JOWO") | (df_html_clean['venue'] == "FOIS")]
df_html_clean.to_csv(path)

NameError: name 'df_html_clean' is not defined

## 3. Extract Abstract and Keyword Information using OLlama and Regex

In [13]:
path = os.path.join("..","..","..", "data", "raw", "conferences","ontology","jowo_fois.csv")
df = pd.read_csv(path)

In [14]:
df.head()

,Unnamed: 0,title,year,authors,doi,url,ee,venue,pages,type
0,54,Encountering the Physical World.,2017,Bahar Aameri; Michael Gruninger,NaN,https://dblp.org/rec/conf/jowo/AameriG17,https://ceur-ws.org/Vol-2050/FOMI_paper_6.pdf,JOWO,NaN,Conference and Workshop Papers
1,55,Business Process Languages: An Ontology-Based ...,2017,Greta Adamo; Stefano Borgo; Chiara Di Francesc...,NaN,https://dblp.org/rec/conf/jowo/AdamoBFGGS17,https://ceur-ws.org/Vol-2050/FOMI_paper_5.pdf,JOWO,NaN,Conference and Workshop Papers
2,56,BPMN 2.0 Choreography Language: Interface or B...,2017,Greta Adamo; Stefano Borgo; Chiara Di Francesc...,NaN,https://dblp.org/rec/conf/jowo/AdamoBFGR17,https://ceur-ws.org/Vol-2050/FOMI_paper_2.pdf,JOWO,NaN,Conference and Workshop Papers
3,57,An Approach to Interaction-Based Concept Conve...,2017,Kemo Adrian; Enric Plaza,NaN,https://dblp.org/rec/conf/jowo/AdrianP17,https://ceur-ws.org/Vol-2050/WINKS_paper_4.pdf,JOWO,NaN,Conference and Workshop Papers
4,58,Talking about Forests: An Example of Sharing I...,2017,Lucía Gómez Álvarez; Brandon Bennett; Adam Ric...,NaN,https://dblp.org/rec/conf/jowo/AlvarezBR17,https://ceur-ws.org/Vol-2050/WINKS_paper_3.pdf,JOWO,NaN,Conference and Workshop Papers


### 3.1 Regex Parser

In [ ]:

extractor = PDFExtractor(use_ollama_fallback=True)  # oder True mit Ollama

# Ab sofort immer so aufrufen:
df_enriched = extractor.run_pipeline(
    df, 
    url_column="ee", 
    delay=0.5,
    save_path="../data/raw/conferences/ontology/jowo_fois_with_abstracts.csv"
)

df_enriched.to_csv("../data/raw/conferences/ontology/jowo_fois_with_abstracts.csv", index=False)
